# Stream Clipper v2 → YouTube Shorts (Colab, GPU)
**Pipeline:** audio-only download → multi-signal hype score (loudness + pitch + YAMNet laughter/shout + optional Twitch chat) → top ~24 candidates → Whisper transcribes *only* those → snap to sentence edges → LLM ranks by hook/humor/emotion and writes titles → download *only* the winning segments → parallel GPU render (split/blur/crop layout, burned-in word captions, loudness-normalized) → zip.

**Runtime:** Runtime → Change runtime type → **T4 GPU**. Run cells top to bottom.
**API key (optional but recommended):** add `GEMINI_API_KEY` (free tier) or `ANTHROPIC_API_KEY` in Colab → 🔑 Secrets.

In [ ]:
!pip -q install -U yt-dlp faster-whisper chat-downloader soundfile tensorflow_hub matplotlib
!apt -qq install -y ffmpeg > /dev/null

## 1. Settings

In [ ]:
VIDEO_URL   = "PASTE_STREAM_VOD_URL_HERE"   # Twitch / YouTube VOD
USE_UPLOAD  = False                          # True = upload a file instead of downloading

# ---- how many / how long
NUM_CLIPS     = 8        # final shorts
N_CANDIDATES  = 24       # audio peaks sent to Whisper + LLM (more = better picks, a bit slower)
MIN_LEN, MAX_LEN = 20, 40
HOOK_PAD      = 4        # seconds before the moment starts rising (hook lands fast)
TAIL          = 10       # seconds after the peak (reaction)
MIN_GAP       = 90       # min seconds between candidates
SKIP_EDGE     = 120      # ignore first/last N seconds (intro/outro)
MAX_HEIGHT    = 720

# ---- look
LAYOUT   = "split"       # "split" = gameplay top + facecam bottom | "blur" | "crop"
FACECAM  = (0.74, 0.62, 0.26, 0.38)   # x, y, w, h as FRACTIONS of the frame (only for "split")
CAPTIONS = True          # burned-in word captions (needs Whisper)

# ---- signals
USE_CHAT = True          # Twitch VOD chat spikes (ignored if unavailable)
USE_YAMNET = True        # laughter / shout / scream classifier
WEIGHTS  = {"loud": 0.40, "pitch": 0.15, "event": 0.30, "chat": 0.15}   # re-normalized if a signal is missing

# ---- models
WHISPER_MODEL = "small"  # "base" faster, "medium"/"large-v3" more accurate
LLM = "none"             # "none" = no API (local ranking) | "gemini" | "anthropic"
REVIEW = False           # True = after ranking, you pick the final clips yourself from a list
GEMINI_MODEL    = "gemini-2.5-flash"
ANTHROPIC_MODEL = "claude-sonnet-5-5"

## 2. Audio only (fast pass 1)
We don't download the full video. Video is fetched later, only for the winning moments.

In [ ]:
import os, subprocess, shutil
os.makedirs("work", exist_ok=True); os.makedirs("clips", exist_ok=True); os.makedirs("caps", exist_ok=True)

if USE_UPLOAD:
    from google.colab import files
    up = files.upload()
    shutil.move(list(up.keys())[0], "work/video.mp4")
    subprocess.run(["ffmpeg","-y","-i","work/video.mp4","-vn","-ac","1","-ar","16000","work/audio.wav"],
                   check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
else:
    for fmt in ("ba", "ba/b"):
        r = subprocess.run(["yt-dlp","-f",fmt,"-x","--audio-format","wav",
                            "--postprocessor-args","ffmpeg:-ac 1 -ar 16000",
                            "-o","work/audio.%(ext)s", VIDEO_URL])
        if r.returncode == 0: break
    else:
        raise RuntimeError("Download failed. Try USE_UPLOAD=True (YouTube often blocks Colab IPs).")
print("audio ready:", os.path.getsize("work/audio.wav")//1_000_000, "MB")

## 3. Score the whole stream
Signals are measured **relative to a 2-minute rolling baseline**, so a quiet streamer's real spike counts as much as a loud streamer's.

In [ ]:
import numpy as np, soundfile as sf, librosa
from scipy.signal import butter, sosfilt, find_peaks
from scipy.ndimage import uniform_filter1d, median_filter

y, sr = sf.read("work/audio.wav", dtype="float32")
if y.ndim > 1: y = y.mean(1)
assert sr == 16000, sr
duration = len(y) / sr
print(f"duration: {duration/60:.1f} min")

hop = sr // 2; step = hop / sr                    # 0.5 s grid
yv = sosfilt(butter(4, [300, 3400], btype="band", fs=sr, output="sos"), y)
rms  = librosa.feature.rms(y=yv, frame_length=sr, hop_length=hop)[0]
cent = librosa.feature.spectral_centroid(y=yv, sr=sr, n_fft=2048, hop_length=hop)[0]
n = min(len(rms), len(cent)); rms, cent = rms[:n], cent[:n]
del yv
t = np.arange(n) * step

z   = lambda x: (x - x.mean()) / (x.std() + 1e-9)
rel = lambda x: x - median_filter(x, size=max(3, int(120/step)), mode="nearest")   # subtract 2-min baseline

signals = {"loud": z(rel(np.log(rms + 1e-6))), "pitch": z(rel(cent))}
print("loudness + pitch done")

In [ ]:
# ---- YAMNet: probability of laughter / shouting / screaming / cheering per moment
if USE_YAMNET:
    import tensorflow as tf, tensorflow_hub as hub, csv
    try:
        for g in tf.config.list_physical_devices("GPU"): tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass
    yam = hub.load("https://tfhub.dev/google/yamnet/1")
    with open(yam.class_map_path().numpy().decode()) as f:
        names = [r["display_name"] for r in csv.DictReader(f)]
    HYPE = {"Screaming","Shout","Yell","Battle cry","Laughter","Belly laugh","Chuckle, chortle","Cheering","Whoop","Gasp"}
    idx = [i for i, nm in enumerate(names) if nm in HYPE]

    CH = 600 * sr; ts, vs = [], []
    for a in range(0, len(y), CH):
        sc, _, _ = yam(tf.constant(y[a:a+CH]))
        v = sc.numpy()[:, idx].max(1)
        ts.append(a/sr + 0.48*np.arange(len(v)) + 0.48); vs.append(v)
    hype = np.interp(t, np.concatenate(ts), np.concatenate(vs))
    signals["event"] = z(hype)
    print("YAMNet done, classes used:", len(idx))

In [ ]:
# ---- Twitch chat replay: messages per 0.5 s (great virality signal)
if USE_CHAT and not USE_UPLOAD and "twitch.tv" in VIDEO_URL:
    try:
        from chat_downloader import ChatDownloader
        chat = np.zeros(n)
        for m in ChatDownloader().get_chat(VIDEO_URL):
            ts_ = m.get("time_in_seconds")
            if ts_ is not None and 0 <= ts_ < duration: chat[min(n-1, int(ts_/step))] += 1
        if chat.sum() > 0:
            signals["chat"] = z(uniform_filter1d(chat, size=int(10/step)))
            print("chat messages:", int(chat.sum()))
    except Exception as e:
        print("chat skipped:", e)

In [ ]:
import matplotlib.pyplot as plt

w = {k: WEIGHTS[k] for k in signals}; s = sum(w.values()); w = {k: v/s for k, v in w.items()}
print("signals/weights:", {k: round(v, 2) for k, v in w.items()})
score = sum(w[k] * signals[k] for k in signals)
score = uniform_filter1d(score, size=int(3/step))

edge = SKIP_EDGE if duration > 4*SKIP_EDGE else 0
score[t < edge] = score.min(); score[t > duration - edge] = score.min()

peaks, _ = find_peaks(score, height=np.percentile(score, 90), distance=int(MIN_GAP/step))
cand_idx = sorted(sorted(peaks, key=lambda p: score[p], reverse=True)[:N_CANDIDATES])

floor = np.percentile(score, 70)
candidates = []
for p in cand_idx:
    j = p
    while j > 0 and (p - j)*step < 15 and score[j-1] > floor: j -= 1      # walk back to where it started rising
    start = max(0, j*step - HOOK_PAD)
    end   = min(duration, p*step + TAIL)
    if end - start < MIN_LEN: end = min(duration, start + MIN_LEN)
    if end - start > MAX_LEN: end = start + MAX_LEN
    candidates.append(dict(peak=p*step, start=start, end=end, audio=float(score[p])))

plt.figure(figsize=(14, 3)); plt.plot(t, score, lw=.6)
plt.scatter([c["peak"] for c in candidates], [c["audio"] for c in candidates], c="r", zorder=3)
plt.xlabel("seconds"); plt.title("Hype score + candidates"); plt.show()
print(len(candidates), "candidates")

## 4. Transcribe candidates only + snap to sentence boundaries

In [ ]:
import difflib
from faster_whisper import WhisperModel

need_words = CAPTIONS or LLM != "none"
if need_words:
    try:
        import torch; gpu = torch.cuda.is_available()
    except Exception: gpu = False
    try:
        wm = WhisperModel(WHISPER_MODEL, device="cuda" if gpu else "cpu", compute_type="float16" if gpu else "int8")
    except Exception as e:
        print("GPU whisper failed, using CPU:", e); wm = WhisperModel(WHISPER_MODEL, device="cpu", compute_type="int8")

    def nearest(vals, target, lo, hi):
        ok = [v for v in vals if lo <= v <= hi]
        return min(ok, key=lambda v: abs(v - target)) if ok else None

    for i, c in enumerate(candidates, 1):
        a = max(0, c["start"] - 6); b = min(duration, c["end"] + 6)
        segs, _ = wm.transcribe(y[int(a*sr):int(b*sr)], word_timestamps=True, vad_filter=True, beam_size=1)
        segs = list(segs)
        words = [(wd.word.strip(), wd.start + a, wd.end + a) for sg in segs for wd in (sg.words or [])]
        ns = nearest([sg.start + a for sg in segs], c["start"], c["start"]-5, c["start"]+3)
        ne = nearest([sg.end + a for sg in segs],   c["end"],   c["end"]-3,   c["end"]+6)
        st, en = (ns if ns is not None else c["start"]), (ne if ne is not None else c["end"])
        if en - st > MAX_LEN or en - st < MIN_LEN: st, en = c["start"], c["end"]      # snapping made it too long/short
        c["start"], c["end"] = st, min(duration, en)
        c["words"] = [w_ for w_ in words if c["start"] - 0.2 <= w_[1] <= c["end"]]
        c["text"] = " ".join(w_[0] for w_ in c["words"])
        print(f"[{i:02d}/{len(candidates)}] {int(c['start']//60):02d}:{int(c['start']%60):02d}  {c['text'][:70]}")
else:
    for c in candidates: c["words"], c["text"] = [], ""

# drop near-duplicate moments (same joke repeated)
keep = []
for c in sorted(candidates, key=lambda c: -c["audio"]):
    if len(c["text"]) > 30 and any(difflib.SequenceMatcher(None, c["text"], k["text"]).ratio() > 0.8 for k in keep): continue
    keep.append(c)
candidates = sorted(keep, key=lambda c: c["start"])
for i, c in enumerate(candidates): c["id"] = i
print(len(candidates), "after dedupe")

## 5. Ranking (no API needed)
Audio finds *loud*; the transcript tells us what is actually *engaging*. With LLM="none" a local scorer judges each transcript: speech density, how fast the hook starts, reaction words, questions/exclamations. Optional LLM and manual review are supported.

In [ ]:
import json, re, requests

def get_key(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        import getpass; return getpass.getpass(f"{name}: ")

PROMPT = """You are a viral YouTube Shorts editor. Below are candidate clips from a livestream, each with its transcript.
Score each 0-10 on: hook (does the first 3 seconds grab attention?), humor, emotion/hype, and standalone clarity (understandable without context).
Penalize clips that are mostly silence, music, or mid-thought.
Return ONLY a JSON list, one object per candidate: {"id": int, "hook": int, "humor": int, "emotion": int, "clarity": int, "title": "<=60 chars, curiosity-driven, no clickbait lies", "hashtags": ["#shorts", ...4 more]}

CANDIDATES:
"""

llm_ok = False
if LLM != "none":
    items = [{"id": c["id"], "duration": round(c["end"]-c["start"]), "transcript": c["text"][:1500] or "(no speech)"} for c in candidates]
    prompt = PROMPT + json.dumps(items, ensure_ascii=False)
    try:
        if LLM == "gemini":
            r = requests.post(f"https://generativelanguage.googleapis.com/v1beta/models/{GEMINI_MODEL}:generateContent",
                headers={"x-goog-api-key": get_key("GEMINI_API_KEY")},
                json={"contents": [{"parts": [{"text": prompt}]}], "generationConfig": {"responseMimeType": "application/json"}}, timeout=120)
            r.raise_for_status(); raw = r.json()["candidates"][0]["content"]["parts"][0]["text"]
        else:
            r = requests.post("https://api.anthropic.com/v1/messages",
                headers={"x-api-key": get_key("ANTHROPIC_API_KEY"), "anthropic-version": "2023-06-01"},
                json={"model": ANTHROPIC_MODEL, "max_tokens": 6000, "messages": [{"role": "user", "content": prompt}]}, timeout=120)
            r.raise_for_status(); raw = r.json()["content"][0]["text"]
        ratings = json.loads(re.sub(r"^```(json)?|```$", "", raw.strip(), flags=re.M).strip())
        by_id = {x["id"]: x for x in ratings}
        for c in candidates:
            x = by_id.get(c["id"])
            if x:
                c["llm"] = (x["hook"]*1.5 + x["humor"] + x["emotion"] + x["clarity"]*0.5) / 40
                c["title"], c["hashtags"] = x.get("title", ""), x.get("hashtags", [])
        llm_ok = True
    except Exception as e:
        print("LLM ranking failed, falling back to audio score only:", e)

HYPE_WORDS = re.compile(r"\b(oh|omg|no way|wtf|what the|holy|bro|dude|insane|crazy|let'?s go|lets go|yes+|no+|hahaha|haha|lol|clutch|dead|bruh|wait|how|why|never|first time|can'?t believe|ez|gg)\b", re.I)

def local_score(c):
    dur = max(1, c["end"] - c["start"]); wd = c["words"]
    if not wd: return 0.0
    wps = len(wd) / dur
    density = min(wps / 3.0, 1.0)                                   # ~3 words/s = energetic talking
    cov = len(set(int(w_[1] - c["start"]) for w_ in wd)) / dur       # fraction of seconds with speech
    early = sum(1 for w_ in wd if w_[1] - c["start"] < 3) >= 3       # speech right at the start = hook
    hits = min(len(HYPE_WORDS.findall(c["text"])) / 5.0, 1.0)
    punct = min((c["text"].count("!") + c["text"].count("?")) / 4.0, 1.0)
    return 0.30*density + 0.25*cov + 0.20*hits + 0.15*punct + 0.10*early

def auto_title(c):
    sents = re.split(r"(?<=[.!?])\s+", c["text"].strip())
    best = max(sents, key=lambda t_: t_.count("!") + t_.count("?") + len(HYPE_WORDS.findall(t_)), default="")
    best = (best or c["text"])[:60].strip()
    return best.rstrip(",;") if len(best) > 8 else f"Stream moment {c['id']+1}"

amin, amax = min(c["audio"] for c in candidates), max(c["audio"] for c in candidates)
for c in candidates:
    a_n = (c["audio"] - amin) / (amax - amin + 1e-9)
    if llm_ok and "llm" in c:
        c["final"] = 0.7*c["llm"] + 0.3*a_n
    else:
        c["final"] = (0.6*a_n + 0.4*local_score(c)) if need_words else a_n
        c["title"] = auto_title(c) if c["text"] else f"Stream moment {c['id']+1}"
    c.setdefault("title", f"Stream moment {c['id']+1}"); c.setdefault("hashtags", ["#shorts", "#gaming", "#streamer"])

final = sorted(sorted(candidates, key=lambda c: -c["final"])[:NUM_CLIPS], key=lambda c: c["start"])
for i, c in enumerate(final, 1):
    print(f"{i}. {int(c['start']//60):02d}:{int(c['start']%60):02d}  {c['final']:.2f}  {c['title']}")

if REVIEW:
    pool = sorted(candidates, key=lambda c: -c["final"])[:NUM_CLIPS*2]
    for k, c in enumerate(pool, 1):
        print(f"\n[{k}] {int(c['start']//60):02d}:{int(c['start']%60):02d}  score {c['final']:.2f}\n    {c['text'][:200]}")
    pick = input("\nNumbers to keep, comma-separated (blank = keep auto picks): ").strip()
    if pick:
        final = sorted([pool[int(x)-1] for x in pick.split(",") if x.strip().isdigit() and 0 < int(x) <= len(pool)], key=lambda c: c["start"])
        print("using", len(final), "clips")

## 6. Render (pass 2: download only winning segments, render in parallel on GPU)

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# ---- encoder: NVENC if the Colab image supports it, else libx264
nv = subprocess.run(["ffmpeg","-y","-f","lavfi","-i","nullsrc=s=256x256","-frames:v","1","-c:v","h264_nvenc","-f","null","-"],
                    capture_output=True).returncode == 0
VENC = ["-c:v","h264_nvenc","-preset","p4","-rc","vbr","-cq","23","-b:v","0"] if nv else ["-c:v","libx264","-preset","veryfast","-crf","23"]
WORKERS = 3 if nv else 2
print("encoder:", "NVENC (GPU)" if nv else "libx264 (CPU)")

def fmt_t(s):
    return f"{int(s//3600)}:{int(s%3600//60):02d}:{s%60:05.2f}"

def make_ass(words, clip_start, path):
    align, marg = {"split": (2, 800), "blur": (2, 560), "crop": (2, 560)}[LAYOUT]
    head = ("[Script Info]\nScriptType: v4.00+\nPlayResX: 1080\nPlayResY: 1920\n\n[V4+ Styles]\n"
            "Format: Name,Fontname,Fontsize,PrimaryColour,SecondaryColour,OutlineColour,BackColour,Bold,Italic,Underline,StrikeOut,ScaleX,ScaleY,Spacing,Angle,BorderStyle,Outline,Shadow,Alignment,MarginL,MarginR,MarginV,Encoding\n"
            f"Style: Cap,DejaVu Sans,78,&H0000FFFF,&H00FFFFFF,&H00000000,&H00000000,-1,0,0,0,100,100,0,0,1,7,2,{align},60,60,{marg},1\n\n"
            "[Events]\nFormat: Layer,Start,End,Style,Name,MarginL,MarginR,MarginV,Effect,Text\n")
    lines, grp = [], []
    def flush():
        if not grp: return
        txt = " ".join(g[0] for g in grp).upper().replace("{", "(").replace("}", ")")
        lines.append((grp[0][1]-clip_start, grp[-1][2]-clip_start, txt)); grp.clear()
    for wd in words:
        if grp and (len(grp) >= 3 or wd[1] - grp[-1][2] > 0.6): flush()
        grp.append(wd)
    flush()
    with open(path, "w", encoding="utf-8") as f:
        f.write(head)
        for k, (a, b, txt) in enumerate(lines):
            if k+1 < len(lines) and lines[k+1][0] - b < 0.3: b = lines[k+1][0]
            f.write(f"Dialogue: 0,{fmt_t(max(0,a))},{fmt_t(max(a+0.2,b))},Cap,,0,0,0,,{txt}\n")

def build_filter(ass=None):
    if LAYOUT == "split":
        fx, fy, fw, fh = FACECAM
        base = ("[0:v]split=2[g][c];"
                "[g]crop=ih*1080/1152:ih,scale=1080:1152[gt];"
                f"[c]crop=iw*{fw}:ih*{fh}:iw*{fx}:ih*{fy},scale=1080:768:force_original_aspect_ratio=increase,crop=1080:768[cb];"
                "[gt][cb]vstack[base]")
    elif LAYOUT == "blur":
        base = ("[0:v]split=2[bg][fg];"
                "[bg]scale=108:192:force_original_aspect_ratio=increase,crop=108:192,boxblur=4:2,scale=1080:1920[b];"
                "[fg]scale=1080:-2[f];[b][f]overlay=(W-w)/2:(H-h)/2[base]")
    else:
        base = "[0:v]scale=-2:1920,crop=1080:1920[base]"
    tail = f"[base]ass={ass},format=yuv420p[v]" if ass else "[base]format=yuv420p[v]"
    return base + ";" + tail

def process(i, c):
    start, end = c["start"], c["end"]; dur = end - start
    if USE_UPLOAD:
        src, off = "work/video.mp4", start
    else:
        seg_s = max(0, start - 2); seg_e = end + 2
        for f in os.listdir("work"):
            if f.startswith(f"seg_{i:02d}."): os.remove(os.path.join("work", f))
        r = subprocess.run(["yt-dlp","-q","-f",f"bv*[height<={MAX_HEIGHT}]+ba/b[height<={MAX_HEIGHT}]",
            "--download-sections", f"*{seg_s:.0f}-{seg_e:.0f}", "--force-keyframes-at-cuts",
            "--merge-output-format","mp4","-o",f"work/seg_{i:02d}.%(ext)s", VIDEO_URL], capture_output=True, text=True)
        src = f"work/seg_{i:02d}.mp4"
        if r.returncode != 0 or not os.path.exists(src): raise RuntimeError("segment download failed: " + r.stderr[-300:])
        off = start - int(seg_s)
    ass = None
    if CAPTIONS and c["words"]:
        ass = f"caps/clip_{i:02d}.ass"; make_ass(c["words"], start, ass)
    out = f"clips/short_{i:02d}.mp4"
    cmd = ["ffmpeg","-y","-ss",f"{off:.2f}","-i",src,"-t",f"{dur:.2f}","-filter_complex",build_filter(ass),
           "-map","[v]","-map","0:a?","-af","loudnorm=I=-14:TP=-1.5:LRA=11"] + VENC + \
          ["-c:a","aac","-b:a","128k","-movflags","+faststart",out]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0: raise RuntimeError("ffmpeg failed: " + r.stderr[-400:])
    return out

results = {}
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = {ex.submit(process, i, c): (i, c) for i, c in enumerate(final, 1)}
    for f in as_completed(futs):
        i, c = futs[f]
        try: results[i] = f.result(); print("made", results[i])
        except Exception as e: print(f"clip {i} FAILED:", e)

## 7. Titles, preview, download

In [ ]:
meta = []
for i, c in enumerate(final, 1):
    if i in results:
        meta.append({"file": results[i], "title": c["title"], "hashtags": " ".join(c["hashtags"]),
                     "source_time": f"{int(c['start']//60):02d}:{int(c['start']%60):02d}", "score": round(c["final"], 3)})
json.dump(meta, open("clips/metadata.json", "w"), indent=2, ensure_ascii=False)
for m in meta: print(m["file"], "|", m["title"], "|", m["hashtags"])

from IPython.display import Video, display
if meta: display(Video(meta[0]["file"], embed=True, width=270))

In [ ]:
!zip -qr clips.zip clips
from google.colab import files
files.download("clips.zip")

---
## Tuning guide
- **Wrong facecam crop?** Adjust `FACECAM` fractions (x, y, w, h). Take a screenshot of the VOD and measure.
- **Clips feel flat?** Raise `N_CANDIDATES` to 30–40 so the LLM has more to choose from; raise `WEIGHTS["event"]` for comedy streams, `["loud"]` for FPS/hype streams.
- **Cuts too late/early?** `HOOK_PAD` (seconds before the rise) and `TAIL` (seconds after the peak).
- **YouTube blocks Colab downloads?** Download on your PC and use `USE_UPLOAD=True`, or mount Drive.
- **No LLM:** ranking uses audio + a local transcript scorer. Set `REVIEW=True` to choose the final clips yourself from the shortlist.
- **Slow Whisper?** Use `WHISPER_MODEL="base"`.
- **Next upgrades:** punch-in zoom at the peak, active-word caption highlight, auto-upload via the YouTube Data API.